#Data Engineering Project: Rapily Implemented End-to-End Scalable AWS ETL Pipeline

This notebook documents an AWS data engineering project that built a historical dataset of SPX/SPXW options quotes with fitted option greeks (delta, gamma, vega, theta, IV) at 5-minute intervals from 2022 to 2026.

**Data source:** [Massive](https://massive.com) REST API  
**Storage:** AWS S3 (Parquet)  
**Compute:** AWS Lambda + EC2 Spot Fleet (via SQS)  
**Orchestration:** AWS Step Functions  
**Project Context & Execution Notes:**
Due to strict time constraints (primarily the cost of the data access per subscription period), development speed was prioritized over optimal code implementation and standard security protocols. Because no sensitive data was involved, risk exposure was minimal. Server-side security was actively monitored throughout execution, and all temporary API keys and credentials were deleted immediately upon project completion. In a professional environment these risks would generally be unacceptable. Additionally, several scripts might have been modified on-the-fly during operational execution so minor alterations may be required to run this code as presented. This notebook is as faithful of a reconstruction as could be compiled.

---

## Pipeline Overview

```
Trading Dates (yfinance)
        │
        ▼
┌─────────────────────┐
│  Stage 1: Symbols   │  Lambda × date  →  metadata/symbols/{date}.json
└─────────────────────┘
        │
        ▼
┌─────────────────────┐
│  Stage 2: Quotes    │  Lambda × 5-min interval  →  opts_quotes_raw/{date}_{HHMM}.parquet
└─────────────────────┘
        │
        ▼  (cleanup pass — see Stage 2b)
┌─────────────────────┐
│  Stage 3: Greeks    │  EC2 worker × file (via SQS)  →  opts_quotes_greeks/{date}_{HHMM}.parquet
└─────────────────────┘
        │
        ▼
┌─────────────────────┐
│  Stage 3b: Compact  │  Downcast dtypes in-place on S3
└─────────────────────┘
```

All files live in a single S3 bucket: `opts-data-2026-03-22`

---
## Stage 1 — Symbol Fetching

### Motivation

The Massive API returns quotes per-ticker. We first need the full list of active SPX contracts for each trading date before we can fetch any quotes. Contracts expiring within 30 days of the target date are included.

### Getting Trading Dates

Trading dates were sourced from Yahoo Finance using `yfinance`. This gives a clean list of actual market open days with no weekends or holidays.

In [ ]:
import yfinance as yf
import pandas as pd
import json

start_date = '2022-04-01'
end_date = '2026-04-29'

stock_data = yf.download(['^SPX'], start=start_date, end=end_date, interval='1d', progress=False)
dates = stock_data.index.strftime('%Y-%m-%d').tolist()

print(f"Total trading dates: {len(dates)}")
print(f"First: {dates[0]}, Last: {dates[-1]}")

### Lambda: `fetch_symbols_lambda`

For each date, a Lambda function queries the Massive API for SPX contracts and stores the ticker list as JSON in S3.

- **Input:** `{ "date_str": "2024-01-15" }`
- **Output:** `s3://.../metadata/symbols/2024-01-15.json` — a JSON array of option ticker strings like `O:SPXW240115C04700000`

In [ ]:
import boto3
import json
from datetime import datetime, timedelta
from massive import RESTClient

BUCKET = 'opts-data-2026-03-22'

def fetch_and_store_symbols(date_str):
    s3 = boto3.client('s3')
    client = RESTClient(api_key='YOUR_API_KEY', pagination=True, trace=False, verbose=True)
    target_date = datetime.strptime(date_str, '%Y-%m-%d')
    max_expiry = (target_date + timedelta(days=30)).strftime('%Y-%m-%d')
    contracts = [
        c.ticker for c in client.list_options_contracts(
            underlying_ticker='SPX',
            as_of=date_str,
            expiration_date_lte=max_expiry,
            expired='false',
            order='asc',
            limit=1000,
            sort='strike_price'
        )
    ]
    s3.put_object(
        Bucket=BUCKET,
        Key=f'metadata/symbols/{date_str}.json',
        Body=json.dumps(contracts)
    )

def lambda_handler(event, context):
    fetch_and_store_symbols(event.get('date_str', '2026-04-05'))
    return {'statusCode': 200}

### Orchestration: Step Functions (Symbol Pass)

A Step Functions `Map` state fans out across all trading dates, invoking `fetch_symbols_lambda` with up to 10 concurrent executions.

```json
{
  "StartAt": "ProcessDates",
  "States": {
    "ProcessDates": {
      "Type": "Map",
      "MaxConcurrency": 10,
      "ItemProcessor": {
        "ProcessorConfig": { "Mode": "INLINE" },
        "StartAt": "InvokeLambda",
        "States": {
          "InvokeLambda": {
            "Type": "Task",
            "Resource": "arn:aws:lambda:us-region-1:1234567890:function:fetch_symbols_lambda",
            "End": true
          }
        }
      },
      "End": true
    }
  }
}
```

---
## Stage 2 — Quote Fetching

### Motivation

With symbol lists in place, we need one bid/ask snapshot per contract per 5-minute interval across the trading day (09:30–16:00 ET). That's 78 intervals × ~4 years of trading days = roughly 80,000 Lambda invocations.

### Generating the Step Functions Input

Each Lambda invocation handles one 5-minute interval for one date. The trigger input is a JSON file listing every `{date, start_ns, end_ns, timestamp}` tuple, generated locally and uploaded to S3.

In [ ]:
import json
import pandas as pd
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

def generate_step_function_input(date_strings):
    eastern = ZoneInfo('America/New_York')
    date_list = []
    for date_str in date_strings:
        current_time = datetime.strptime(date_str, '%Y-%m-%d').replace(hour=9, minute=30, tzinfo=eastern)
        end_market = current_time.replace(hour=16, minute=0)
        while current_time < end_market:
            interval_end = current_time + timedelta(minutes=5)
            date_list.append({
                'as_of': date_str,
                'start': int(current_time.timestamp() * 1_000_000_000),
                'end': int(interval_end.timestamp() * 1_000_000_000),
                'timestamp': f"{date_str}_{current_time.strftime('%H%M')}"
            })
            current_time = interval_end
    return {'date_list': date_list}

# Example: one day produces 78 intervals
sample = generate_step_function_input(['2024-01-15'])
print(f"Intervals for one day: {len(sample['date_list'])}")
print(f"First: {sample['date_list'][0]}")
print(f"Last:  {sample['date_list'][-1]}")

### Lambda: `fetch_quotes_lambda`

Each invocation:
1. Reads the symbol list for the date from S3
2. Fires 40 concurrent threads via `ThreadPoolExecutor`, one per ticker
3. Fetches the first available quote in the 5-minute window from the Massive API
4. Writes the result to `s3://.../opts_quotes_raw/{date}_{HHMM}.parquet`

In [ ]:
import pandas as pd
import boto3
import json
import io
import urllib3
from massive import RESTClient
from concurrent.futures import ThreadPoolExecutor

BUCKET = 'opts-data-2026-03-22'

def lambda_handler(event, context):
    s3 = boto3.client('s3')
    client = RESTClient(api_key='YOUR_API_KEY', pagination=False, num_pools=50)
    client.client = urllib3.PoolManager(num_pools=50, maxsize=50, block=False)
    obj = s3.get_object(Bucket=BUCKET, Key=f"metadata/symbols/{event['as_of']}.json")
    tickers = json.loads(obj['Body'].read().decode('utf-8'))
    def fetch_quote(ticker):
        try:
            for q in client.list_quotes(ticker=ticker, timestamp_gte=event['start'], timestamp_lt=event['end'], order='asc', limit=1, sort='timestamp'):
                return {'symbol': ticker, 'ask': q.ask_price, 'bid': q.bid_price, 'ask size': q.ask_size, 'bid size': q.bid_size, 'sip_timestamp': q.sip_timestamp}
        except Exception as e:
            print(f'Error fetching {ticker}: {e}')
        return None
    with ThreadPoolExecutor(max_workers=40) as executor:
        results = list(executor.map(fetch_quote, tickers))
    df = pd.DataFrame([r for r in results if r is not None])
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False, engine='pyarrow', compression='snappy')
    s3.put_object(Bucket=BUCKET, Key=f"opts_quotes_raw/{event['timestamp']}.parquet", Body=buffer.getvalue())
    return {'statusCode': 200}

### Orchestration: Step Functions (Quote Pass) — v1 vs v2

The Step Functions definition went through two versions.

**v1 (inline mode, 10 concurrency):** The full `date_list` was passed directly in the Step Functions input payload. This hit payload size limits for large date ranges.

**v2 (distributed mode, 30 concurrency):** The state machine reads its input directly from the S3 JSON file, avoiding payload limits entirely. `ToleratedFailureCount: 1000` was added to allow partial failures without aborting the entire run — important given the scale.

```json
{
  "StartAt": "ProcessQuotes",
  "States": {
    "ProcessQuotes": {
      "Type": "Map",
      "ItemReader": {
        "Resource": "arn:aws:states:::s3:getObject",
        "ReaderConfig": { "InputType": "JSON", "ItemsPointer": "/date_list" },
        "Parameters": {
          "Bucket": "opts-data-2026-03-22",
          "Key": "step_function_input.json"
        }
      },
      "MaxConcurrency": 30,
      "ItemProcessor": {
        "ProcessorConfig": { "Mode": "DISTRIBUTED", "ExecutionType": "STANDARD" },
        "StartAt": "InvokeLambda",
        "States": {
          "InvokeLambda": {
            "Type": "Task",
            "Resource": "arn:aws:states:::lambda:invoke",
            "Parameters": {
              "FunctionName": "arn:aws:lambda:us-region-1:1234567890:function:fetch_quotes_lambda",
              "Payload.$": "$"
            },
            "End": true
          }
        }
      },
      "ToleratedFailureCount": 1000,
      "End": true
    }
  }
}
```

> **Reflection:** Step Functions turned out to be more infrastructure than the job needed. It worked, but managing state machine definitions and input payloads added friction. For the greek fitting stage, the simpler SQS + EC2 worker pattern (see Stage 3) was much easier to operate.

### S3 Output Structure After Stage 2

```
opts-data-2026-03-22/
├── metadata/
│   └── symbols/
│       ├── 2022-03-07.json
│       ├── 2022-03-08.json
│       └── ...  (~1,000 files)
└── opts_quotes_raw/
    ├── 2022-03-07_0930.parquet
    ├── 2022-03-07_0935.parquet
    └── ...  (~80,000 files)
```

Each raw parquet file has schema: `symbol | ask | bid | ask size | bid size | sip_timestamp`

---
## Stage 2b — Raw File Cleanup

After the initial quote run, a significant number of files were corrupted, empty, or missing. This stage was the messiest part of the project — many scripts were written iteratively as different failure modes were discovered.

### Problem 1: Early Market Close Days

The market closes at 1:00 PM ET on certain days (Black Friday, day before July 4th, Christmas Eve). All 5-minute intervals after 1 PM were legitimately empty. These produced zero-row parquet files.

| Date | Occasion | Empty Range |
|---|---|---|
| 2022-11-25 | Black Friday | 13:20 – 15:55 |
| 2023-07-03 | Day before July 4th | 13:20 – 15:55 |
| 2023-11-24 | Black Friday | 13:20 – 15:55 |
| 2024-07-03 | Day before July 4th | 13:15 – 15:55 |
| 2024-11-29 | Black Friday | 13:20 – 15:55 |
| 2024-12-24 | Christmas Eve | 13:15 – 15:55 |
| 2025-07-03 | Day before July 4th | 13:15 – 15:55 |
| 2025-11-28 | Black Friday | 13:15 – 15:55 |
| 2025-12-24 | Christmas Eve | 13:15 – 15:55 |

A handful of isolated gaps existed on other dates too (e.g. 2022-06-03, 2023-10-25), likely due to API outages.

### Problem 2: Corrupted Files

Some Lambda invocations wrote files that were structurally invalid parquet (incomplete writes from timeouts or API errors). These caused downstream Athena queries to fail entirely.

### Discovery: Athena

The bad files were first identified by pointing Athena at `opts_quotes_raw/` and querying for anomalously small row counts.

In [ ]:
# Athena DDL to create the table over the raw parquet prefix
athena_create = """
CREATE EXTERNAL TABLE IF NOT EXISTS options_audit.quotes_raw (
  `symbol`        STRING,
  `ask`           DOUBLE,
  `bid`           DOUBLE,
  `ask size`      BIGINT,
  `bid size`      BIGINT,
  `sip_timestamp` BIGINT
)
STORED AS PARQUET
LOCATION 's3://opts-data-2026-03-22/opts_quotes_raw/';
"""

# Query to find files with suspiciously low row counts
# (run after moving known-corrupt files out first, as corrupted files cause this query to fail)
athena_find_tiny = """
WITH summary AS (
    SELECT
        element_at(split("$path", '/'), -1) AS filename,
        COUNT(*) AS record_count
    FROM options_audit.quotes_raw
    GROUP BY "$path"
),
quantiles AS (
    SELECT *, PERCENT_RANK() OVER (ORDER BY record_count ASC) AS p_rank
    FROM summary
)
SELECT filename, record_count
FROM quantiles
WHERE p_rank <= 0.25
ORDER BY record_count ASC;
"""

print("Athena queries defined (run these in the AWS Athena console or via boto3)")

### Validation: Fast Parquet Footer Check

Rather than downloading full multi-MB files, we only fetch the last 4KB of each small file. The Parquet format stores its metadata footer at the end of the file, so `pyarrow` can validate the file and get the row count from just those 4KB.

In [ ]:
import boto3
import pyarrow.parquet as pq
from botocore.config import Config
from concurrent.futures import ThreadPoolExecutor
from io import BytesIO

BUCKET = 'opts-data-2026-03-22'
s3 = boto3.client('s3', config=Config(max_pool_connections=50))

def check_file(bucket, key):
    """Returns (key, 'EMPTY'|'CORRUPTED') or None if the file is valid."""
    try:
        response = s3.get_object(Bucket=bucket, Key=key, Range='bytes=-4096')
        metadata = pq.read_metadata(BytesIO(response['Body'].read()))
        if metadata.num_rows == 0:
            return (key, 'EMPTY')
    except Exception:
        return (key, 'CORRUPTED')
    return None

def find_bad_files(bucket, prefix, size_threshold=5120):
    """Scan files under prefix that are smaller than size_threshold bytes."""
    bad_files = []
    paginator = s3.get_paginator('list_objects_v2')
    with ThreadPoolExecutor(max_workers=20) as executor:
        futures = []
        for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
            for obj in page.get('Contents', []):
                if obj['Size'] <= size_threshold:
                    futures.append(executor.submit(check_file, bucket, obj['Key']))
        for future in futures:
            result = future.result()
            if result:
                bad_files.append(result)
    return bad_files

# results = find_bad_files(BUCKET, 'opts_quotes_raw/')
# for file_key, error_type in results:
#     print(f"{error_type}: {file_key}")

### Moving Bad Files Out

Bad files were moved to `corrupted_files/` or `empty_files/` prefixes (S3 copy + delete) so they wouldn't block Athena queries or the greek fitting stage.

In [ ]:
import boto3

def move_bad_files(bucket, bad_files):
    s3 = boto3.client('s3')
    for file_key, error_type in bad_files:
        dest_prefix = 'empty_files' if error_type == 'EMPTY' else 'corrupted_files'
        new_key = f"{dest_prefix}/{file_key.split('/')[-1]}"
        s3.copy_object(Bucket=bucket, CopySource={'Bucket': bucket, 'Key': file_key}, Key=new_key)
        s3.delete_object(Bucket=bucket, Key=file_key)
        print(f"Moved {file_key} → {new_key}")

# As a quicker alternative for bulk-moving sub-1KB files (likely corrupt):
def move_tiny_files(bucket, src, dst, limit=1000):
    s3 = boto3.client('s3')
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=src):
        for obj in page.get('Contents', []):
            if obj['Size'] < limit:
                old_key = obj['Key']
                new_key = f"{dst}{old_key.split('/')[-1]}"
                s3.copy_object(Bucket=bucket, CopySource={'Bucket': bucket, 'Key': old_key}, Key=new_key)
                s3.delete_object(Bucket=bucket, Key=old_key)
                print(f"Moved: {old_key}")

### Re-running Failed Intervals

After identifying which dates had bad files, the master Step Functions input JSON was filtered to only those dates and the Lambda was re-triggered for those intervals.

The master input file (`step_function_input_full.json`) was too large to load into memory directly, so `ijson` was used for streaming parsing.

In [ ]:
import ijson
import json

bad_day_list = [
    '2022-06-02', '2022-06-03', '2022-11-25', '2023-07-03', '2023-10-25',
    '2023-11-24', '2024-07-03', '2024-11-29', '2024-12-24', '2025-07-03',
    '2025-08-06', '2025-10-22', '2025-11-28', '2025-12-24'
]

# Stream through the large JSON to extract only intervals for bad days
retry_items = []
with open('step_function_input_full.json', 'rb') as f:
    for item in ijson.items(f, 'date_list.item'):
        if item['as_of'] in bad_day_list:
            retry_items.append(item)

print(f"Intervals to retry: {len(retry_items)}")

# Write a targeted retry input and re-trigger Step Functions
with open('step_function_input_retry.json', 'w') as f:
    json.dump({'date_list': retry_items}, f, indent=2)

---
## Stage 3 — Greek Fitting

### Motivation

The raw quote files contain bid/ask prices but no option greeks. For each 5-minute snapshot we need to:
1. Fit an arbitrage-free implied volatility surface across all strikes and expiries (using the SVI model)
2. Derive delta, gamma, vega, theta, and model IV from that surface

A modified version of the `sanos` library (from [Hans Buehler's deephedging repo](https://github.com/hansbuehler/deephedging)) provides the SVI surface fitting implementation. This process is not shown or detailed here. Check the github repository and it's associated research paper if you're curious about this.

### Why Not Lambda?

The first attempt was to run greek fitting inside Lambda functions, mirroring the quote-fetching architecture. This failed due to **deployment package size limits**.

The combined uncompressed dependency footprint was ~274 MB:

| Package | Size (MB) |
|---|---|
| pandas | 48 |
| numpy | 32 |
| scipy | 94 |
| plotly | 40 |
| boto3 | 1 |
| sanos (smoothvol + bs) | <1 |
| **Total** | **~274** |

Lambda's 250 MB unzipped limit (across function + layers) made this unworkable even after splitting into separate layers (`plotly_layer.zip`, `sanos_layer.zip`). Step Functions was also reconsidered here — the overhead of defining state machines was more friction than the workload warranted.

### Solution: SQS + EC2 Spot Fleet

The architecture that worked:

1. Enqueue all `opts_quotes_raw/*.parquet` S3 keys into an SQS queue
2. Launch a fleet of EC2 spot instances (`t4g.small`, ARM)
3. Each instance boots, installs dependencies, pulls `worker.py` from S3, and polls the queue until empty
4. Instances shut themselves down on completion

This required no Step Functions state machines, no Lambda layers, and the spot pricing kept compute costs very low.

### SQS Queue Setup

In [ ]:
import boto3

sqs = boto3.client('sqs', region_name='us-region-1')
s3 = boto3.client('s3', region_name='us-region-1')

BUCKET = 'opts-data-2026-03-22'
QUEUE_URL = 'https://sqs.us-region-1.amazonaws.com/1234567890/Greek-Processing-Queue'

# Create the queue (one-time)
sqs.create_queue(
    QueueName='Greek-Processing-Queue',
    Attributes={'VisibilityTimeout': '300', 'MessageRetentionPeriod': '86400'}
)

# Fill queue with only unprocessed files (diff raw vs greeks prefix)
processed_keys = set()
paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=BUCKET, Prefix='opts_quotes_greeks/'):
    for obj in page.get('Contents', []):
        processed_keys.add(obj['Key'].replace('opts_quotes_greeks/', '', 1))

batch = []
for page in paginator.paginate(Bucket=BUCKET, Prefix='opts_quotes_raw/'):
    for obj in page.get('Contents', []):
        filename = obj['Key'].replace('opts_quotes_raw/', '', 1)
        if filename not in processed_keys and filename:
            batch.append({'Id': str(len(batch) % 10), 'MessageBody': obj['Key']})
            if len(batch) == 10:
                sqs.send_message_batch(QueueUrl=QUEUE_URL, Entries=batch)
                batch = []
if batch:
    sqs.send_message_batch(QueueUrl=QUEUE_URL, Entries=batch)

print('Queue populated')

### EC2 Spot Fleet: Provisioning

Instances are launched with a `UserData` bootstrap script that fully provisions each VM from scratch, runs `worker.py`, and then terminates. No AMI baking required.

In [ ]:
import boto3

ssm = boto3.client('ssm', region_name='us-region-1')
ec2 = boto3.client('ec2', region_name='us-region-1')

BUCKET = 'opts-data-2026-03-22'

latest_ami = ssm.get_parameter(
    Name='/aws/service/ami-amazon-linux-latest/al2023-ami-kernel-default-arm64'
)['Parameter']['Value']

user_data_script = f"""#!/bin/bash
# Swap (sanos vol fitting is memory-intensive)
dd if=/dev/zero of=/swapfile bs=128M count=16
chmod 600 /swapfile && mkswap /swapfile && swapon /swapfile
echo "/swapfile swap swap defaults 0 0" >> /etc/fstab

# Dependencies
dnf update -y && dnf install -y python3.12 git
python3.12 -m ensurepip --upgrade
python3.12 -m pip install boto3 pandas==2.2.2 numpy==2.0.2 plotly==5.24.1 scipy==1.16.3 cdxcore==0.1.79 cvxpy==1.6.7 watchtower

# sanos library (SVI vol surface fitting)
git clone --depth 1 https://github.com/hansbuehler/deephedging temp_repo
mkdir -p sanos && cp temp_repo/tmp_sanos/*.py ./sanos/ && rm -rf temp_repo
sed -i 's/from nbs.utils.cdxcore2.verbose import Context/from cdxcore.verbose import Context/g' sanos/bs.py

# Pull worker and rate data from S3
aws s3 cp s3://{BUCKET}/worker.py worker.py
aws s3 cp s3://{BUCKET}/rates_2022-2026.csv rates_2022-2026.csv

python3.12 worker.py
shutdown -h now
"""

ec2.run_instances(
    UserData=user_data_script,
    MaxCount=6,
    MinCount=1,
    ImageId=latest_ami,
    InstanceType='t4g.small',
    InstanceInitiatedShutdownBehavior='terminate',
    KeyName='key_2026-03-22',
    BlockDeviceMappings=[{'DeviceName': '/dev/xvda', 'Ebs': {'DeleteOnTermination': True, 'VolumeSize': 8, 'VolumeType': 'standard'}}],
    NetworkInterfaces=[{'AssociatePublicIpAddress': True, 'DeviceIndex': 0, 'Groups': ['sg-1234567890']}],
    CreditSpecification={'CpuCredits': 'unlimited'},
    IamInstanceProfile={'Arn': 'arn:aws:iam::1234567890:instance-profile/DataProcessorRole'},
    InstanceMarketOptions={'MarketType': 'spot', 'SpotOptions': {'InstanceInterruptionBehavior': 'terminate', 'SpotInstanceType': 'one-time'}},
    MetadataOptions={'HttpEndpoint': 'enabled', 'HttpPutResponseHopLimit': 2, 'HttpTokens': 'required'}
)

print('Spot fleet launched')

### Worker: Greek Fitting Logic

Each EC2 instance runs `worker.py` in a loop, polling SQS for the next file key, processing it, and deleting the message on success. Failures are logged to CloudWatch and the message is left in the queue (visible again after the visibility timeout).

The key steps inside `process_file()`:

In [ ]:
# Abbreviated version of worker.py showing the core fitting logic

import pandas as pd
import boto3
import io
import numpy as np
from scipy.stats import norm
from sanos.smoothvol import ExpiryData, SmoothCallSurface, SmoothCallSurfaceConfig
from sanos.bs import remove_eps_strikes, bs_call, bs_implied, bs_vega, bs_dk
import gc
import logging, socket, watchtower

BUCKET = 'opts-data-2026-03-22'
QUEUE_URL = 'https://sqs.us-region-1.amazonaws.com/1234567890/Greek-Processing-Queue'

logger = logging.getLogger('worker')
logger.setLevel(logging.INFO)
logger.addHandler(watchtower.CloudWatchLogHandler(
    log_group_name='Greek-Processor',
    log_stream_name=f'worker-{socket.gethostname()}',
    boto3_client=boto3.client('logs', region_name='us-region-1')
))

def process_file(s3_key):
    s3 = boto3.client('s3', region_name='us-region-1')
    parquet_file = s3.get_object(Bucket=BUCKET, Key=s3_key)
    df = pd.read_parquet(io.BytesIO(parquet_file['Body'].read()))

    # --- Parse OCC ticker symbol ---
    pattern = r'O:(?P<ticker>[A-Z]+)\d?(?P<expiration>\d{6})(?P<type>[CP])(?P<strike_raw>\d{8})'
    extracted = df['symbol'].str.extract(pattern)
    extracted['strike'] = extracted['strike_raw'].astype(float) / 1000
    df = pd.concat([df, extracted[['ticker', 'expiration', 'type', 'strike']]], axis=1)
    df = df[df['ticker'] == 'SPXW']  # Focus on SPX weekly options

    # --- Time calculations ---
    df['snapshot_time'] = int(pd.to_datetime(df['sip_timestamp'], unit='ns').min().timestamp())
    df['quote_unixtime'] = (df['sip_timestamp'].values.view('int64') // 10**9).astype(int)
    df['mid'] = (df['bid'] + df['ask']) / 2
    df['expiration'] = pd.to_datetime(df['expiration'], format='%y%m%d')
    df['expire_unix'] = (
        (df['expiration'] + pd.Timedelta(hours=16))
        .dt.tz_localize('America/New_York', ambiguous='infer')
        .dt.tz_convert('UTC').dt.tz_localize(None)
        .values.astype('datetime64[ns]').view('int64') // 10**9
    )
    df['DTE'] = ((df['expire_unix'] - df['quote_unixtime']) / 86400.0).round(4)
    df['T_years'] = df['DTE'] / 365.0
    df = df.rename(columns={'type': 'putcall'})

    # --- Load risk-free rates ---
    rates_df = pd.read_csv('rates_2022-2026.csv', parse_dates=['date'])
    # (rates are merged onto df by date to get the appropriate discount rate per row)

    # --- SVI Vol Surface Fitting (per expiry) ---
    # SmoothCallSurface fits an arbitrage-free SVI surface across all strikes
    # for each expiry date in the snapshot, then prices calls and derives greeks
    # via Black-Scholes using the fitted IV.
    #
    # Output columns added:
    #   model_price  → arb-free mid price from fitted surface
    #   model_iv     → fitted implied volatility (SVI)
    #   model_delta  → dV/dS
    #   model_gamma  → d²V/dS²
    #   model_vega   → dV/dσ
    #   model_theta  → dV/dt

    # [Fitting loop omitted for brevity — see worker.py]

    df = df.rename(columns={
        'model_price': 'arb_free_price',
        'model_iv': 'iv_svi',
        'model_delta': 'delta',
        'model_gamma': 'gamma',
        'model_vega': 'vega',
        'model_theta': 'theta'
    })

    # --- Write output ---
    input_filename = s3_key.split('/')[-1].replace('.parquet', '')
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False, engine='pyarrow', compression='snappy')
    s3.put_object(Bucket=BUCKET, Key=f'opts_quotes_greeks/{input_filename}.parquet', Body=buffer.getvalue())
    logger.info(f'Completed: {input_filename}')
    return True

def main():
    sqs = boto3.client('sqs', region_name='us-region-1')
    while True:
        response = sqs.receive_message(QueueUrl=QUEUE_URL, MaxNumberOfMessages=1, WaitTimeSeconds=20)
        if 'Messages' not in response:
            break
        message = response['Messages'][0]
        try:
            if process_file(message['Body']):
                sqs.delete_message(QueueUrl=QUEUE_URL, ReceiptHandle=message['ReceiptHandle'])
        except Exception:
            logger.error(f"Failed: {message['Body']}", exc_info=True)
        gc.collect()

if __name__ == '__main__':
    main()

### Monitoring

Workers log to CloudWatch log group `Greek-Processor`, with one stream per instance hostname. Progress could be tracked by watching message count on the SQS queue or querying CloudWatch Logs Insights:

```
# CloudWatch Logs Insights query to find errors
filter @message like /\d\d:/
| stats count() by @logStream
```

A diagnostic HTML report was also generated per file and uploaded to `greek_reports/{date}_{HHMM}.html`, containing plotly vol surface charts and a metrics table showing fit quality (% of contracts within bid-ask spread, max pricing error).

---
## Stage 3b — Output Compaction

The initial greek output files used default pandas dtypes (float64, int64, object). After profiling memory usage, significant savings were available by downcasting.

A batch script (`reduction_script2.py`) ran over all files in `opts_quotes_greeks/`, rewriting each one in-place with tighter types and dropping redundant columns.

Early runs of this script hit a `'symbol'` KeyError on files written under the old column schema (before the `rename_dict` was finalized in `worker.py`). Those files were re-processed by the worker.

In [ ]:
import pandas as pd
import boto3
import io
import logging
from concurrent.futures import ThreadPoolExecutor

logging.basicConfig(filename='greek_downcasting.log', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

BUCKET = 'opts-data-2026-03-22'
PREFIX = 'opts_quotes_greeks/'
TICKER_PATTERN = r'O:(?P<ticker>[A-Z]+)\d?(?P<expiration>\d{6})(?P<type>[CP])(?P<strike_raw>\d{8})'
COLS_TO_DROP = ['symbol', 'quote_unixtime', 'quote_unixtime_real', 'snapshot_time',
                'date', 'time', 'DTE', 'DTE_unix', 'mid', 'expire_unix']
TARGET_COLS = ['ask', 'bid', 'arb_free_price', 'delta', 'gamma', 'vega', 'theta', 'iv_svi']

s3 = boto3.client('s3')

def process_file(key):
    try:
        obj = s3.get_object(Bucket=BUCKET, Key=key)
        df = pd.read_parquet(io.BytesIO(obj['Body'].read()))
        df['expiration'] = df['symbol'].str.extract(TICKER_PATTERN)['expiration'].astype('int32')
        df['ticker'] = df['ticker'].astype('category')
        df['putcall'] = df['putcall'].astype('category')
        df['strike'] = df['strike'].astype('float32')
        df['ask size'] = df['ask size'].astype('int32')
        df['bid size'] = df['bid size'].astype('int32')
        df = df.drop(columns=COLS_TO_DROP)
        df[TARGET_COLS] = df[TARGET_COLS].astype('float32')
        out_buffer = io.BytesIO()
        df.to_parquet(out_buffer, index=False, engine='pyarrow', compression='snappy')
        s3.put_object(Bucket=BUCKET, Key=key, Body=out_buffer.getvalue())
        logging.info(key)
        return (True, key)
    except Exception as e:
        return (False, f'{key}: {str(e)}')

def run_batch():
    paginator = s3.get_paginator('list_objects_v2')
    keys = [obj['Key'] for page in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX)
            for obj in page.get('Contents', []) if obj['Key'].endswith('.parquet')]
    logging.info(f'Starting batch process for {len(keys)} files.')
    with ThreadPoolExecutor(max_workers=10) as executor:
        for i, (success, info) in enumerate(executor.map(process_file, keys)):
            if not success:
                logging.error(f'FAILED: {info}')
            if i % 1000 == 0:
                logging.info(f'Progress: {i}/{len(keys)}')
    logging.info('Batch process complete.')

# run_batch()

---
## Final S3 Structure

```
opts-data-2026-03-22/
├── metadata/
│   └── symbols/
│       └── {date}.json                  ← list of active SPX option tickers per date
├── opts_quotes_raw/
│   └── {date}_{HHMM}.parquet            ← raw bid/ask snapshots
├── opts_quotes_greeks/
│   └── {date}_{HHMM}.parquet            ← quotes + fitted greeks + IV (final output)
├── greek_reports/
│   └── {date}_{HHMM}.html              ← per-snapshot vol surface diagnostic charts
├── corrupted_files/                     ← bad files moved out of raw prefix
└── empty_files/                         ← early-close and gap files
```

### Final Output Schema (`opts_quotes_greeks/`)

| Column | Type | Description |
|---|---|---|
| `ticker` | category | Underlying (SPXW) |
| `expiration` | int32 | Expiry date as YYMMDD |
| `putcall` | category | 'C' or 'P' |
| `strike` | float32 | Strike price |
| `ask` | float32 | Ask price |
| `bid` | float32 | Bid price |
| `ask size` | int32 | Ask size |
| `bid size` | int32 | Bid size |
| `sip_timestamp` | int64 | Quote timestamp (nanoseconds) |
| `arb_free_price` | float32 | Model mid price (SVI surface) |
| `iv_svi` | float32 | Implied volatility from fitted surface |
| `delta` | float32 | dV/dS |
| `gamma` | float32 | d²V/dS² |
| `vega` | float32 | dV/dσ |
| `theta` | float32 | dV/dt |

---
## Reading the Final Data

Files can be read individually or queried across the full dataset using DuckDB directly against S3.

In [ ]:
# Read a single snapshot
import boto3
import pandas as pd
import io

BUCKET = 'opts-data-2026-03-22'
s3 = boto3.client('s3')

def read_snapshot(date_str, time_str):
    key = f'opts_quotes_greeks/{date_str}_{time_str}.parquet'
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    return pd.read_parquet(io.BytesIO(obj['Body'].read()))

df = read_snapshot('2024-01-15', '1000')
print(df.dtypes)
print(df.head())

In [ ]:
# Query across all files via DuckDB + S3
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='us-region-1';")
con.execute("SET s3_access_key_id='YOUR_ACCESS_KEY';")
con.execute("SET s3_secret_access_key='YOUR_SECRET_KEY';")

result = con.execute("""
    SELECT
        putcall,
        expiration,
        strike,
        AVG(iv_svi) AS avg_iv,
        AVG(delta) AS avg_delta
    FROM read_parquet('s3://opts-data-2026-03-22/opts_quotes_greeks/2024-01-15_1000.parquet')
    GROUP BY putcall, expiration, strike
    ORDER BY expiration, strike
""").df()

print(result)